# Marts

Built final analytical tables for KPI calculations, segmentation, retention.  

**Architecture:** three tables linked through common keys.

```
mart_enrollments_detail ──course_id──→ mart_dim_courses
mart_enrollments_detail ──student_id─→ mart_dim_students
```

**Tables:**
- `mart_enrollments_detail` — fact table: each enrollment with its payment
- `mart_dim_courses`        — course dimension table with instructor attributes
- `mart_dim_students`       — student dimension table 


In [ ]:
import duckdb
import pandas as pd
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

con = duckdb.connect()

STG   = 'data/staging'
MARTS = 'data/marts'
os.makedirs(MARTS, exist_ok=True)

def q(sql):
    return con.sql(sql).df()

def save(sql, name):
    df = con.sql(sql).df()
    path = f'{MARTS}/{name}.csv'
    df.to_csv(path, index=False)
    return df


### Mart_enrollments_detail
- LEFT JOIN between `enrollments` and `payments` on `student_id` and `course_id`.    
- Calculation of gross and net revenue.

In [ ]:
mart_enrollments = save("""
SELECT
    e.enrollment_id,
    e.student_id,
    e.course_id,
    e.enroll_date,
    e.enroll_month,
    e.enroll_year,

    -- Payment for this enrollment (net_amount = amount for success, 0 for failed/refunded)
    COALESCE(p.net_amount, 0)   AS payment_amount,

    -- gross_amount = absolute amount regardless of status
    COALESCE(p.amount, 0)       AS gross_amount,

    p.status                    AS payment_status,
    p.payment_date,
    p.payment_month,
    p.payment_year,

    -- Boolean flags
    COALESCE(p.is_success,  FALSE) AS is_success,
    COALESCE(p.is_refunded, FALSE) AS is_refunded,
    COALESCE(p.is_failed,   FALSE) AS is_failed


FROM stg_enrollments e
LEFT JOIN stg_payments p
       ON e.student_id = p.student_id
      AND e.course_id  = p.course_id
""", 'mart_enrollments_detail')

### Mart_dim_courses
- LEFT JOIN between `courses` and `instructors` on `instructor_id`.    

In [ ]:
mart_courses = save("""
SELECT
    c.course_id,
    c.title,
    c.category,
    c.price,
    c.price_tier,
    c.duration_hours,

    i.instructor_name,
    i.tier         AS instructor_tier,
    i.subject_area

FROM stg_courses c
LEFT JOIN stg_instructors i ON c.instructor_id = i.instructor_id
""", 'mart_dim_courses')


### Mart_dim_students
- Aggregated all payment history by students вased on the tables `payments` and `students`.  
- Calculated first and last payment dates, and how many days the student was active.
- Segmented students by purchase behavior.

In [ ]:
mart_students = save("""
WITH

-- ─────────────── Aggregate all payments at the student level ──────────────────────
student_payments AS (
    SELECT
        student_id,
        
        -- Count of successful payment transactions
        SUM(CAST(is_success  AS INTEGER))                           AS successful_payments,

        -- Unique purchased courses (successful payments only)
        COUNT(DISTINCT CASE WHEN is_success THEN course_id END)     AS courses_purchased,

        -- Timeline of when the student was active
        MIN(CASE WHEN is_success THEN payment_date END)             AS first_payment_date,
        MAX(CASE WHEN is_success THEN payment_date END)             AS last_payment_date,
        DATEDIFF('day',
            MIN(CASE WHEN is_success THEN payment_date END),
            MAX(CASE WHEN is_success THEN payment_date END)
        )                                                           AS days_active

    FROM stg_payments
    GROUP BY student_id
),

-- ─────────────── Calculate days to the second purchase ──────────────────────
-- Cross-sell metric: how quickly a one-time buyer returns
purchase_sequence AS (
    SELECT
        student_id,
        payment_date,
        ROW_NUMBER() OVER (
            PARTITION BY student_id
            ORDER BY payment_date
        ) AS purchase_num
    FROM stg_payments
    WHERE is_success = TRUE
),

days_to_second AS (
    SELECT
        first_buy.student_id,
        DATEDIFF('day',
            first_buy.payment_date,
            second_buy.payment_date
        ) AS days_to_second_purchase
    FROM purchase_sequence first_buy
    LEFT JOIN purchase_sequence second_buy
           ON first_buy.student_id  = second_buy.student_id
          AND second_buy.purchase_num = 2
    WHERE first_buy.purchase_num = 1
)

-- ─────────────── Final table: student + payment summary ──────────────────────
SELECT
    s.student_id,
    s.reg_date,
    s.cohort_month,
    s.reg_year,
    s.country,
    s.acquisition_channel,

    
    COALESCE(p.courses_purchased,   0) AS courses_purchased,
    p.first_payment_date,
    p.last_payment_date,
    COALESCE(p.days_active, 0)         AS days_active,
    d.days_to_second_purchase,

    -- Segmentation by purchase behavior (no_purchase < one_time < repeat < power)
    CASE
        WHEN COALESCE(p.successful_payments, 0) = 0 THEN 'no_purchase'
        WHEN COALESCE(p.courses_purchased,   0) = 1 THEN 'one_time'
        WHEN COALESCE(p.courses_purchased,   0) <= 4 THEN 'repeat'
        ELSE                                              'power'
    END AS student_segment

FROM stg_students s
LEFT JOIN student_payments p  ON s.student_id = p.student_id
LEFT JOIN days_to_second   d  ON s.student_id = d.student_id
ORDER BY ltv_net DESC
""", 'mart_dim_students')
